# 29. A three-step funnel, in order, within seven days

**Track:** sql | **Difficulty:** hard | **Tags:** funnel analysis, event sequences, range join, conversion

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("029-ordered-funnel-seven-days")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "events": """
UEFSMRUEFaAHFfIDTBV0FQASAADQAwQlAAkBACAJBwQALg0IACINCAApDQgAGw0IAAwNCAAjDQgEOgAJAQAfCQcEACgNCAAGDQgA
Jw0IAAQNCAAXDQgAMw0IABYNCAAcDQgAOA0IADYNCAAZDQgANw0IAAMNCAANDQgACg0IABENCAA1DQgANA0IAAINCAAvDQgABQ0I
ACENCAAODQgAFQ0IBB0ACQEAFAkHBAAIDQgAGA0IACQNCAAxDQgAKw0IAA8NCAA5DQgACw0IACwNCAAJDQgALQ0IACYNCAAQDQgA
MA0IACoNCAAaDQgABw0IAB4NCAABDQgAMg0IPBIAAAAAAAAAEwAAAAAAAAAVABWaAhWiAiwV1AIVEBUGFQYcGAg6AAAAAAAAABgI
AQAAAAAAAAAWACgIOgAAAAAAAAAYCAEAAAAAAAAAEREAAACNAfCMAwAAANQCAQYtQCAMRBEYwIEkirAwjfNAkTQJlFFZ1wBhTpYN
2sZt3eV9VtUVYGiA4pgBQ0WWXjGZJ5qO6gqZGROpCcFGXNISq1tobyCIa/ekHxyz8qgyc8ZpdG0tNiXV7FiRSER/oM1RQLs4rHTj
Ij5RF7ZKBlacSFfE70UFEF0bjpt3NPZIig0AAAAAFQQVchV0TBUKFQASAAA5kAgAAABwdXJjaGFzZQsAAABhZGRfdG9fY2FydAYA
AABzZWFyY2gBJTx3aXNobGlzdAQAAAB2aWV3FQAVlgEVnAEsFdQCFRAVBhUGHDYAKAh3aXNobGlzdBgLYWRkX3RvX2NhcnQREQAA
AEvwSgMAAADUAgEDLQAQSWMYhgwoJBKSjMIWCiJHkBAZJAwYMKQWksyQMQQJjgg5EFMiEmIwgkmAUCQZghRJUgM4JAiJhNK4JAo1
hgQAABUEFZAVFfARTBXSAhUAEgAAyApQAC+tB6ovBgAAOCNNTjAGAAAgywfmARAM3l++rwEIDOYUZocBCAw3fv2PAQgMm0NiPQEo
DJVKbxEBCAx/iVyDARgMYXWzTwEIDPZnWmwBCAxTufFEAQgMEgpTlwEIDBWeANoBCAzodK4JATgMfKa3WQEQDOWfMOcBCAw29q2k
AQgM6fIXbQEIDK5c+3gBCAz3/oIjATAM8NEqNwEIDHr9HPkFGAg6YL8BCAy8Z7RqAQgICiICBUgMTWjTbgEQDCvS2lsBCCiHEE7G
LwYAAGb0fQWIDNMDT0oBEAxFrjhTAQgMYnIuuAEIDJuWmqMBCAzHsAmFAQgMTXvtuQEIDLNdZoABCAjM8kQF2AjSheUlIAgUGvIF
qAx/jk/HASAMM3UarQEICFkiHSVQCCOkxiU4DBnjwFgBGAgFQSYFMAwBYrifARAMYpewiQEIDNlM6yEB2AzB/dl8ARAM3Nay3AEI
DMhQZv8BCAyNEnW1AQgMuVtM8AEIDBfTIYIBCAgYvYIFSAhptuEF8Aip1PwFYAxkLuNjASAIz7mbBYAI747sBZAIyAWwJQAI2hlM
BcgM4zHBXQEoDEmqpvMBCAwXUYHCAQgMtBQexQEIDNJEFQcBmAzPZmMhAQgMZclMmgEYDMSj70UBCAxHiy5RAQgMy/GIhAEICGev
diVwDDhfKuoBECxz+jkiMAYAALsgNV8BEAy2jwKRAQgIC5c3BZgML0e1+AEQDI6t3usBCAhVVRcFuAx1KmiGARAM34X8aAEIDACL
cc0BCAyTbMdOAQgIdP82JTgIU4ZIBcgMs3VzDwFoDD77/icBCAwg8Im3ASgMxVqnigEIDOnp46kBCAwnjRrdAQgIRV9BBTgM0Om/
yAEQDGMfAHUBCAzevNydAQgMKaKs9wEIDNMIQo4BCAysyrviAQgMuHHr0gEICG/UVgVADOf502ABEAy9bWOWAQgInMr/BcAMqd7i
9QEQDFNBNw0BkAxZ6oCNARAIK5EKJeAMD0P8ewEQDMvD0u8BCAyRSsDxAQgM13xqZgEIDEw+VDYBOAjNtyNFmAz9IDmgARgIk/1A
ZUAIYa1PBYAIC8rgJVgIr0tPBRgIqaqHRWgMYRiVYQEwDOL5Gh4BSAwUZajVARAM1XaxFwEQCDH1QgXoDPU8KW8BGAxaGqXOAQgM
YsYY4wEICBgMeiWADMA0urEBEAyoqn01ATgMhKsI3QEQCF/egCUwDPatHZABEAzm+Bd5AQgIAZEgZZgIGE1KBcgM5cml/AEYDOA3
wWkBCAy8lhzEAQgISBhxZVAIPjzPRWAMQ99pXAEYCDmTj0WoCKQ6lmkABLHFBegMxAr0uwEgDCgsxZIBCAwe0sPZAQgM4aR0CwGY
DBfK7b4BEAwezdCVAQgMvJsPCAEYCIQfggUQDNPs838BGAjZu3ElqAx3RO3kARAIvvqlRegI9SDbJcgI12AcJUAM0BM13gEgDHZ6
G1UBCAz+2zFkAQgM/TeUagEICOhAUwWYOBwVXWkvBgAAnSmM/i8GABUAFfICFfoCLBXUAhUQFQYVBhwYCAA4I01OMAYAGAgAU7nx
RC8GABYAKAgAOCNNTjAGABgIAFO58UQvBgAREQAAALkB8LgDAAAA1AIBCC0AAQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0e
HyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hp
amtsbW5vcHFyc3R1dncKeHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqAAAAAAAABUEGUw1
ABgGc2NoZW1hFQYAFQQlAhgHdXNlcl9pZAAVDCUCGApldmVudF90eXBlJQBMHAAAABUEJQIYCGV2ZW50X3RzJRRMjBIcLAAAAAAA
FtQCGRwZPCYAHBUEGTUABhAZGAd1c2VyX2lkFQIW1AIW3goWuAcmmgQmCBwYCDoAAAAAAAAAGAgBAAAAAAAAABYAKAg6AAAAAAAA
ABgIAQAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA1AIAAAAmABwVDBk1AAYQGRgKZXZlbnRfdHlwZRUCFtQCFoYDFo4D
JtAIJsAHHDYAKAh3aXNobGlzdBgLYWRkX3RvX2NhcnQREQAZLBUEFQAVAgAVABUQFQIAPBbcEhkGGSYA1AIAAAAmABwVBBk1AAYQ
GRgIZXZlbnRfdHMVAhbUAhaoGRaQFibgHCbOChwYCAA4I01OMAYAGAgAU7nxRC8GABYAKAgAOCNNTjAGABgIAFO58UQvBgAREQAZ
LBUEFQAVAgAVABUQFQIAPCkGGSYA1AIAAAAWjCcW1AImCBbWIAAZHBgMQVJST1c6c2NoZW1hGMACLy8vLy8rZ0FBQUFRQUFBQUFB
QUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDQUFBQUFQQUFBQUFRQUFBQ2Mv
Ly8vQUFBQkNoQUFBQUFnQUFBQUJBQUFBQUFBQUFBSUFBQUFaWFpsYm5SZmRITUFBQVlBQ0FBR0FBWUFBQUFBQUFJQTBQLy8vd0FB
QVFVUUFBQUFJQUFBQUFRQUFBQUFBQUFBQ2dBQUFHVjJaVzUwWDNSNWNHVUFBQVFBQkFBRUFBQUFFQUFVQUFnQUJnQUhBQXdBQUFB
UUFCQUFBQUFBQUFFQ0VBQUFBQ0FBQUFBRUFBQUFBQUFBQUFjQUFBQjFjMlZ5WDJsa0FBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFB
QUEAGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAACQDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "029-ordered-funnel-seven-days"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

events = spark.read.parquet((_data_dir / "events.parquet").resolve().as_posix())
events.createOrReplaceTempView("events")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'first_cart_ts': '2d40ce47ee17e499494ca7a8b0eb39db1776649105280cf5e185da8c8127a7a3',
                         'first_purchase_ts': '10fd03387834946c4a7a44a3ad92c51e671ab57c5306235a20c56960b2871b87',
                         'first_view_ts': '4da5854166e8e6e11c20d5a2cccd629293a68cddbdcf8ee047f71637f5d22c4e',
                         'user_id': '7d3c7c6a6bf35eceb8a7e9c7c2701e55465be0e3c8c147872ef6b24f5b880e36'},
 'columns': [['first_cart_ts', 'timestamp_ntz'],
             ['first_purchase_ts', 'timestamp_ntz'],
             ['first_view_ts', 'timestamp_ntz'],
             ['user_id', 'bigint']],
 'fingerprint': 'f173d074330d6683f5bd06211d6aca4b06a72faf36abefc4341589eabccae706',
 'order_matters': False,
 'row_count': 50}

check = make_check(EXPECTED)

## The situation

The growth team measures a three-step funnel: a user views a product, adds something
to the cart, and buys. They want to know how many users get through each step within
a week of first looking.

The existing query counts, for each step, the users who have at least one event of
that type. That overstates every step after the first. Some users buy through a saved
link without using the cart, some events are out of order, and some purchases come
weeks after the view.

## Input tables

### `events`

One row per event.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | The user. |
| `event_type` | STRING | `view`, `add_to_cart`, `purchase`, and others such as `search` and `wishlist`. |
| `event_ts` | TIMESTAMP_NTZ | When the event happened. |

## Requirement

Return one row for every user who has at least one `view`, showing how far through
the funnel they got.

- `first_view_ts` is the user's earliest `view`.
- `first_cart_ts` is the user's earliest `add_to_cart` that is **at or after**
  `first_view_ts` and no more than 7 days after it. NULL if there is none.
- `first_purchase_ts` is the user's earliest `purchase` that is **at or after**
  `first_cart_ts` and no more than 7 days after `first_view_ts`. NULL if there is
  none, and NULL whenever `first_cart_ts` is NULL.
- "No more than 7 days after" includes the moment exactly 7 days later.
- Events before the first view, and events of other types, are ignored.

## Expected output

One row per user with at least one view.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | |
| `first_view_ts` | TIMESTAMP_NTZ | |
| `first_cart_ts` | TIMESTAMP_NTZ | NULL if the user did not reach this step. |
| `first_purchase_ts` | TIMESTAMP_NTZ | NULL if the user did not reach this step. |

In [ ]:
events.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `events`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)